**Phase 3: Stream Processing**

**[1] Import Libraries**

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim
import pandas as pd


**[2] Start Spark Session**

In [3]:
spark = SparkSession.builder.appName("MedicalAppointmentsStreamProcessing").getOrCreate()


**[3] Load Excel Data**

In [4]:
from google.colab import files
uploaded = files.upload()

Saving Updated_Medical_Appointments_Corrected.xlsx to Updated_Medical_Appointments_Corrected.xlsx


In [5]:
xls_path = "Updated_Medical_Appointments_Corrected.xlsx"

df_patients = pd.read_excel(xls_path, sheet_name="Patients")
df_appointments = pd.read_excel(xls_path, sheet_name="Appointments")
df_conditions = pd.read_excel(xls_path, sheet_name="MedicalConditions")

**[4] Convert DataFrames from Pandas to Spark**

In [8]:
patients_sdf = spark.createDataFrame(df_patients)
appointments_sdf = spark.createDataFrame(df_appointments)
conditions_sdf = spark.createDataFrame(df_conditions)


**[5] Merge the three data using PatientId**

In [9]:
merged_df = appointments_sdf \
    .join(patients_sdf, on="PatientId", how="inner") \
    .join(conditions_sdf, on="PatientId", how="inner")


**[6] Filter out patients who did not show up for appointments.**

In [10]:
no_show_df = merged_df.filter(lower(trim(col("No_show"))) == "yes")


**[7] View absent patient data**

In [12]:
no_show_df.select(
    "PatientId",
    "Gender",
    "AppointmentID",
    "ScheduledDay",
    "AppointmentDay",
    "Diabetes"
).show(truncate=False)


+---------+------+-------------+-------------------+-------------------+--------+
|PatientId|Gender|AppointmentID|ScheduledDay       |AppointmentDay     |Diabetes|
+---------+------+-------------+-------------------+-------------------+--------+
|P320     |F     |A320         |2025-01-14 07:00:00|2025-01-15 07:00:00|1       |
|P204     |F     |A204         |2025-01-09 11:00:00|2025-01-10 11:00:00|1       |
|P444     |F     |A444         |2025-01-19 11:00:00|2025-01-20 11:00:00|1       |
|P604     |F     |A604         |2025-01-26 03:00:00|2025-01-27 03:00:00|1       |
|P324     |F     |A324         |2025-01-14 11:00:00|2025-01-15 11:00:00|1       |
|P552     |F     |A552         |2025-01-23 23:00:00|2025-01-24 23:00:00|1       |
|P124     |F     |A124         |2025-01-06 03:00:00|2025-01-07 03:00:00|1       |
|P72      |F     |A72          |2025-01-03 23:00:00|2025-01-04 23:00:00|1       |
|P96      |F     |A96          |2025-01-04 23:00:00|2025-01-05 23:00:00|1       |
|P496     |F    